<a href="https://colab.research.google.com/github/Nifal123/financial-/blob/main/02_Portfolio_Analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Notebook 2 - Portfolio Analytics and Risk Metrics

Author: Your Name
Date: May 2026

What This Notebook Covers
- Annualized return and volatility for each asset
- Sharpe, Sortino and Calmar ratios
- Maximum Drawdown analysis
- Value at Risk and Conditional VaR
- Rolling risk metrics over time


In [1]:
!pip install yfinance plotly -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

save_path = '/content/drive/MyDrive/portfolio_optimizer/'

prices          = pd.read_csv(f'{save_path}prices.csv', index_col=0, parse_dates=True)
daily_returns   = pd.read_csv(f'{save_path}daily_returns.csv', index_col=0, parse_dates=True)
monthly_returns = pd.read_csv(f'{save_path}monthly_returns.csv', index_col=0, parse_dates=True)

RISK_FREE_RATE = 0.05
TRADING_DAYS   = 252

print("All libraries loaded and data imported successfully")
print("Assets:", list(daily_returns.columns))
print("Date range:", daily_returns.index[0].date(), "to", daily_returns.index[-1].date())

Mounted at /content/drive
All libraries loaded and data imported successfully
Assets: ['US Equities', 'Intl Equities', 'Long Bonds', 'Agg Bonds', 'Gold', 'Commodities', 'Real Estate']
Date range: 2018-01-03 to 2024-12-30


In [2]:
def calculate_metrics(returns_df, risk_free_rate=RISK_FREE_RATE):

    metrics = {}
    daily_rf = risk_free_rate / TRADING_DAYS

    for asset in returns_df.columns:
        r = returns_df[asset].dropna()

        ann_return   = r.mean() * TRADING_DAYS
        total_return = (1 + r).prod() - 1
        ann_vol      = r.std() * np.sqrt(TRADING_DAYS)

        downside_returns = r[r < daily_rf]
        downside_dev     = downside_returns.std() * np.sqrt(TRADING_DAYS)

        cumulative  = (1 + r).cumprod()
        rolling_max = cumulative.cummax()
        drawdown    = (cumulative - rolling_max) / rolling_max
        max_drawdown= drawdown.min()

        sharpe  = (ann_return - risk_free_rate) / ann_vol if ann_vol != 0 else 0
        sortino = (ann_return - risk_free_rate) / downside_dev if downside_dev != 0 else 0
        calmar  = ann_return / abs(max_drawdown) if max_drawdown != 0 else 0

        var_95  = np.percentile(r, 5)
        var_99  = np.percentile(r, 1)
        cvar_95 = r[r <= var_95].mean()
        cvar_99 = r[r <= var_99].mean()

        skewness = r.skew()
        kurtosis = r.kurtosis()

        metrics[asset] = {
            'Ann. Return %'     : round(ann_return * 100, 2),
            'Total Return %'    : round(total_return * 100, 2),
            'Ann. Volatility %' : round(ann_vol * 100, 2),
            'Sharpe Ratio'      : round(sharpe, 3),
            'Sortino Ratio'     : round(sortino, 3),
            'Calmar Ratio'      : round(calmar, 3),
            'Max Drawdown %'    : round(max_drawdown * 100, 2),
            'VaR 95% Daily %'   : round(var_95 * 100, 2),
            'CVaR 95% Daily %'  : round(cvar_95 * 100, 2),
            'VaR 99% Daily %'   : round(var_99 * 100, 2),
            'CVaR 99% Daily %'  : round(cvar_99 * 100, 2),
            'Skewness'          : round(skewness, 3),
            'Excess Kurtosis'   : round(kurtosis, 3),
        }

    return pd.DataFrame(metrics).T

metrics_df = calculate_metrics(daily_returns)
print("Metrics calculated successfully")
print(metrics_df.to_string())

Metrics calculated successfully
               Ann. Return %  Total Return %  Ann. Volatility %  Sharpe Ratio  Sortino Ratio  Calmar Ratio  Max Drawdown %  VaR 95% Daily %  CVaR 95% Daily %  VaR 99% Daily %  CVaR 99% Daily %  Skewness  Excess Kurtosis
US Equities             1.25            7.68               6.15        -0.610         -0.766         0.067          -18.58            -0.55             -0.86            -0.92             -1.54    -0.930           32.904
Intl Equities           5.56           31.29              18.17         0.031          0.039         0.163          -34.19            -1.66             -2.67            -2.97             -4.80    -0.862           13.431
Long Bonds             10.39           92.27              14.32         0.377          0.526         0.472          -22.00            -1.48             -2.10            -2.41             -3.24    -0.244            2.865
Agg Bonds               6.69           32.32              23.03         0.073          0

In [3]:
colors = ['#2196F3','#FF9800','#4CAF50','#9C27B0','#F44336','#00BCD4','#FF5722']

fig = go.Figure()

for i, asset in enumerate(metrics_df.index):
    fig.add_trace(go.Scatter(
        x    = [metrics_df.loc[asset, 'Ann. Volatility %']],
        y    = [metrics_df.loc[asset, 'Ann. Return %']],
        mode = 'markers+text',
        name = asset,
        text = [asset],
        textposition = 'top center',
        marker = dict(size=16, color=colors[i], line=dict(width=2, color='white')),
        hovertemplate = f"<b>{asset}</b><br>Volatility: %{{x:.1f}}%<br>Return: %{{y:.1f}}%<extra></extra>"
    ))

fig.add_hline(
    y=RISK_FREE_RATE * 100,
    line_dash='dash',
    line_color='gray',
    annotation_text='Risk-Free Rate (5%)',
    annotation_position='right'
)

fig.update_layout(
    title      = dict(text='Risk vs Return by Asset Class (2018-2024)', font=dict(size=18)),
    xaxis_title= 'Annualized Volatility (%)',
    yaxis_title= 'Annualized Return (%)',
    height     = 550,
    template   = 'plotly_white',
    showlegend = False
)

fig.show()

In [4]:
fig = go.Figure()

bar_colors  = ['#2196F3', '#4CAF50', '#FF9800']
ratio_names = ['Sharpe Ratio', 'Sortino Ratio', 'Calmar Ratio']

for i, ratio in enumerate(ratio_names):
    fig.add_trace(go.Bar(
        name         = ratio,
        x            = metrics_df.index,
        y            = metrics_df[ratio],
        marker_color = bar_colors[i],
        text         = metrics_df[ratio].round(2),
        textposition = 'outside'
    ))

fig.add_hline(y=1.0, line_dash='dash', line_color='red',
              annotation_text='Sharpe 1 = Good', annotation_position='right')

fig.update_layout(
    title      = dict(text='Risk-Adjusted Return Ratios by Asset Class', font=dict(size=18)),
    xaxis_title= 'Asset Class',
    yaxis_title= 'Ratio Value',
    barmode    = 'group',
    height     = 500,
    template   = 'plotly_white',
    legend     = dict(orientation='h', y=1.05, x=0.3)
)

fig.show()

In [8]:
import datetime

fig = go.Figure()

for i, asset in enumerate(daily_returns.columns):
    r           = daily_returns[asset].dropna()
    cumulative  = (1 + r).cumprod()
    rolling_max = cumulative.cummax()
    drawdown    = (cumulative - rolling_max) / rolling_max * 100

    fig.add_trace(go.Scatter(
        x    = drawdown.index,
        y    = drawdown.values,
        name = asset,
        line = dict(width=1.5, color=colors[i])
    ))

# Add the vertical line without an annotation
fig.add_vline(x='2020-03-23', line_dash='dash', line_color='red')

# Add the annotation separately
fig.add_annotation(
    x='2020-03-23',
    y=0.95, # Position the annotation at 95% of the y-axis height (paper coordinates)
    xref='x', yref='paper',
    text='COVID Crash',
    showarrow=False,
    xanchor='center',
    yanchor='bottom',
    font=dict(color='red', size=12)
)

fig.update_layout(
    title      = dict(text='Drawdown from Peak - All Assets (2018-2024)', font=dict(size=18)),
    xaxis_title= 'Date',
    yaxis_title= 'Drawdown (%)',
    height     = 520,
    template   = 'plotly_white',
    hovermode  = 'x unified',
    legend     = dict(orientation='h', y=-0.2)
)

fig.show()

In [9]:
window = 252
fig    = go.Figure()

for i, asset in enumerate(daily_returns.columns):
    r = daily_returns[asset].dropna()

    rolling_sharpe = (
        r.rolling(window).mean() * TRADING_DAYS - RISK_FREE_RATE
    ) / (r.rolling(window).std() * np.sqrt(TRADING_DAYS))

    fig.add_trace(go.Scatter(
        x    = rolling_sharpe.index,
        y    = rolling_sharpe.values,
        name = asset,
        line = dict(width=2, color=colors[i])
    ))

fig.add_hline(y=0, line_color='black', line_width=1)
fig.add_hline(y=1, line_dash='dash', line_color='green',
              annotation_text='Sharpe = 1', annotation_position='right')

fig.update_layout(
    title      = dict(text='Rolling 12-Month Sharpe Ratio (2018-2024)', font=dict(size=18)),
    xaxis_title= 'Date',
    yaxis_title= 'Rolling Sharpe Ratio',
    height     = 520,
    template   = 'plotly_white',
    hovermode  = 'x unified',
    legend     = dict(orientation='h', y=-0.2)
)

fig.show()

In [10]:
investment = 1_000_000

var_data = pd.DataFrame({
    'VaR 95% ($)'  : (metrics_df['VaR 95% Daily %'] / 100 * investment).abs().round(0),
    'CVaR 95% ($)' : (metrics_df['CVaR 95% Daily %'] / 100 * investment).abs().round(0),
    'VaR 99% ($)'  : (metrics_df['VaR 99% Daily %'] / 100 * investment).abs().round(0),
})

fig = go.Figure()

bar_colors2 = ['#FFC107', '#FF5722', '#9C27B0']
col_names   = ['VaR 95% ($)', 'CVaR 95% ($)', 'VaR 99% ($)']

for i, col in enumerate(col_names):
    fig.add_trace(go.Bar(
        name         = col,
        x            = var_data.index,
        y            = var_data[col],
        marker_color = bar_colors2[i],
        text         = var_data[col].apply(lambda x: f'${x:,.0f}'),
        textposition = 'outside'
    ))

fig.update_layout(
    title      = dict(text='Value at Risk on $1,000,000 Portfolio - Daily Loss Estimates', font=dict(size=17)),
    xaxis_title= 'Asset Class',
    yaxis_title= 'Potential Daily Loss ($)',
    barmode    = 'group',
    height     = 520,
    template   = 'plotly_white',
    legend     = dict(orientation='h', y=1.05, x=0.25)
)

fig.show()

In [11]:
metrics_df.to_csv(f'{save_path}asset_metrics.csv')

best_sharpe = metrics_df['Sharpe Ratio'].idxmax()
best_return = metrics_df['Ann. Return %'].idxmax()
lowest_vol  = metrics_df['Ann. Volatility %'].idxmin()

print("All results saved to Google Drive")
print("")
print("KEY FINDINGS")
print("============")
print(f"Best Risk-Adjusted Return : {best_sharpe}  (Sharpe = {metrics_df.loc[best_sharpe,'Sharpe Ratio']})")
print(f"Highest Raw Return        : {best_return}  ({metrics_df.loc[best_return,'Ann. Return %']}%)")
print(f"Lowest Volatility         : {lowest_vol}  ({metrics_df.loc[lowest_vol,'Ann. Volatility %']}%)")
print("")
print("Notebook 2 Complete - Next: Notebook 3 Optimization Engine")

All results saved to Google Drive

KEY FINDINGS
Best Risk-Adjusted Return : Gold  (Sharpe = 0.501)
Highest Raw Return        : Gold  (14.75%)
Lowest Volatility         : US Equities  (6.15%)

Notebook 2 Complete - Next: Notebook 3 Optimization Engine


In [12]:
# ============================================================
# CELL 9: Save results for use in later notebooks
# ============================================================

metrics_df.to_csv(f'{save_path}asset_metrics.csv')

print("✅ Analytics complete and saved!")
print(f"   📁 Saved: asset_metrics.csv")

print("\n" + "="*60)
print("📊 SUMMARY — KEY FINDINGS")
print("="*60)

best_sharpe  = metrics_df['Sharpe Ratio'].idxmax()
best_return  = metrics_df['Ann. Return %'].idxmax()
lowest_vol   = metrics_df['Ann. Volatility %'].idxmin()
least_dd     = metrics_df['Max Drawdown %'].idxmax()

print(f"\n   🏆 Best Risk-Adjusted Return : {best_sharpe}")
print(f"      Sharpe = {metrics_df.loc[best_sharpe,'Sharpe Ratio']}")

print(f"\n   📈 Highest Raw Return        : {best_return}")
print(f"      Return = {metrics_df.loc[best_return,'Ann. Return %']}%")

print(f"\n   🛡️  Lowest Volatility         : {lowest_vol}")
print(f"      Volatility = {metrics_df.loc[lowest_vol,'Ann. Volatility %']}%")

print(f"\n   ⚓ Smallest Max Drawdown      : {least_dd}")
print(f"      Max DD = {metrics_df.loc[least_dd,'Max Drawdown %']}%")

print("\n🎉 Notebook 2 Complete!")
print("   ➡️  Next: Notebook 3 — Optimization Engine (Efficient Frontier)")

✅ Analytics complete and saved!
   📁 Saved: asset_metrics.csv

📊 SUMMARY — KEY FINDINGS

   🏆 Best Risk-Adjusted Return : Gold
      Sharpe = 0.501

   📈 Highest Raw Return        : Gold
      Return = 14.75%

   🛡️  Lowest Volatility         : US Equities
      Volatility = 6.15%

   ⚓ Smallest Max Drawdown      : US Equities
      Max DD = -18.58%

🎉 Notebook 2 Complete!
   ➡️  Next: Notebook 3 — Optimization Engine (Efficient Frontier)
